In [85]:
from dotenv import load_dotenv
load_dotenv()

True

In [86]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Literal


#Libs for Agent
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent
from langchain.tools import tool

In [87]:
class FlowState(BaseModel):
    question:str = Field(description="USer ask the question")
    category: str = Literal['coding','google_search','weather']
    answer: str = Field(default="")

In [88]:
# to get structure output from llm
class QuestionCategory(BaseModel):
    category: Literal['coding','google_search','weather'] = Field(default='google_search', description="Question Category")

In [89]:
llm = ChatGroq(model='openai/gpt-oss-20b')

In [90]:
#defining your agents - googleSearchAgent, Weather Agent

search = GoogleSerperAPIWrapper()
tools=[search.run]

google_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a agent and can search for any question on google."
)

#weather agent 
@tool
def get_weather(city: str):
    """
        This is a function that return the temprature of city.
        Args:
            city: Name of city
    """
    return f"The current temprature of {city} is 23.C"


weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are the agent and you jobs is to find the temparature of given country"
)

In [91]:
def check_question_category(state:FlowState) -> FlowState:
    st_llm = llm.with_structured_output(QuestionCategory)

    res = st_llm.invoke(f"I want to know the category of my question, question is: {state.question}")
    state.category = res.category
    # print(state)
    return state
    

In [92]:
flow = FlowState(question="Some of two number in python")

cdp = check_question_category(flow)
cdp

FlowState(question='Some of two number in python', category='coding', answer='')

In [93]:
def route(state:FlowState) -> Literal['coding','google_search','weather']:
    return state.category

In [94]:
def coding_node(state:FlowState) -> FlowState:
    res = llm.invoke(f"You are a coding expert : {state.question}")
    state.answer = res.content
    return state
    

In [95]:
def weather_node(state:FlowState) -> FlowState:
    res = weather_agent.invoke({"message":[{"role":"user","content":state.question}]})
    state.answer = res["messages"][-1].content
    return state

In [96]:
def google_search_node(state:FlowState)-> FlowState:
    res = google_agent.invoke({
        "messages": [{"role":"user","containt":state.question}]
    })
    state.amswer = res["messages"][-1].content
    return state

Creating Graph

In [100]:
graph = StateGraph(FlowState)

graph.add_node("check_question_category",check_question_category)
graph.add_node("coding",coding_node)
graph.add_node("weather",weather_node)
graph.add_node("google_search", google_search_node)

graph.add_edge(START, "check_question_category")
graph.add_conditional_edges("check_question_category",route)
graph.add_edge("weather", END)
graph.add_edge("coding", END)
graph.add_edge("google_search",END)

graph = graph.compile()

In [104]:
question= """write a code in python sum of two number

"""
res = graph.invoke({"question": question})
res

{'question': 'write a code in python sum of two number\n\n',
 'category': 'coding',
 'answer': 'Here’s a minimal, ready‑to‑run Python script that asks the user for two numbers and prints their sum.  \nFeel free to copy‑paste it into a file (e.g., `sum_two_numbers.py`) and run it with `python sum_two_numbers.py`.\n\n```python\n# sum_two_numbers.py\n"""\nSimple program that reads two numbers from the user,\ncalculates their sum, and prints the result.\n"""\n\ndef get_number(prompt: str) -> float:\n    """\n    Prompt the user for a number and return it as a float.\n    Keeps asking until a valid numeric input is provided.\n    """\n    while True:\n        try:\n            return float(input(prompt))\n        except ValueError:\n            print("That’s not a valid number. Please try again.")\n\ndef main() -> None:\n    print("=== Sum of Two Numbers ===")\n    a = get_number("Enter the first number: ")\n    b = get_number("Enter the second number: ")\n    total = a + b\n    print(f"The